In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.pipeline import Pipeline

In [2]:
np.random.seed(42)

In [6]:
X = np.linspace(0, 5, 100).reshape(-1, 1)
y = 3 * X**2 - 2 * X + 1 + np.random.randn(100, 1) * 4

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (80, 1)
Test size: (20, 1)


In [39]:
poly=PolynomialFeatures(degree=10)


In [40]:
X_train_poly=poly.fit_transform(X_train)
X_test_poly=poly.transform(X_test)

In [41]:
lr=LinearRegression()

In [42]:
lr.fit(X_train_poly, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [43]:
train_r2_score=r2_score(y_train,lr.predict(X_train_poly))
test_r2_score=r2_score(y_test,lr.predict(X_test_poly))

In [44]:
print(train_r2_score)
print(test_r2_score)

0.9721623013574844
0.9606539987501822


In [45]:
train_r2_score-test_r2_score

0.011508302607302179

In [46]:
ridge=Ridge(alpha=1.0)
ridge.fit(X_train_poly,y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [47]:
train_r2_ridge = r2_score(y_train, ridge.predict(X_train_poly))
test_r2_ridge  = r2_score(y_test,  ridge.predict(X_test_poly))


In [48]:
train_r2_ridge-test_r2_ridge

0.0011675680221594487

In [71]:
lasso = Lasso(alpha=0.1,max_iter=10000)
lasso.fit(X_train_poly, y_train)

D:\anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.534e+02, tolerance: 3.445e+00
  model = cd_fast.enet_coordinate_descent(


,alpha,0.1
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,10000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [72]:
train_r2_lasso = r2_score(y_train, lasso.predict(X_train_poly))
test_r2_lasso  = r2_score(y_test,  lasso.predict(X_test_poly))


In [73]:
print(f"Lasso Regression (degree=10, alpha=0.1)")
print(f"Train R²: {train_r2_lasso:.4f}")
print(f"Test  R²: {test_r2_lasso:.4f}")
print(f"Gap: {train_r2_lasso - test_r2_lasso:.4f}")


Lasso Regression (degree=10, alpha=0.1)
Train R²: 0.9684
Test  R²: 0.9714
Gap: -0.0031


In [74]:

zero_coefs = np.sum(lasso.coef_ == 0)

In [75]:
zero_coefs

np.int64(3)

In [76]:
from sklearn.model_selection import GridSearchCV

In [77]:
ridge_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=10)),
    ('ridge', Ridge())
])

In [78]:
alphas = {'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100]}


In [79]:
grid_ridge=GridSearchCV(ridge_pipeline,alphas,cv=5,scoring='r2')

In [80]:
grid_ridge.fit(X_train,y_train)

D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.49095e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.29656e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.67261e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.20206e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.68051e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
D:\anaconda\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning:

,estimator,"Pipeline(step...e', Ridge())])"
,param_grid,"{'ridge__alpha': [0.001, 0.01, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,degree,10


In [81]:

print(f"Best alpha (Ridge): {grid_ridge.best_params_}")
print(f"Test R²: {r2_score(y_test, grid_ridge.predict(X_test)):.4f}")

Best alpha (Ridge): {'ridge__alpha': 1}
Test R²: 0.9686


In [94]:
# Cell 29 update karo
lasso_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=10)),
    ('lasso', Lasso(max_iter=50000))   # 10000 → 50000
])

In [95]:
alphas_lasso = {'lasso__alpha': [0.001, 0.01, 0.1, 1, 10]}


In [96]:
grid_lasso=GridSearchCV(lasso_pipeline,alphas_lasso,cv=5,scoring='r2')

In [97]:
grid_lasso.fit(X_train,y_train)

D:\anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.646e+02, tolerance: 2.700e+00
  model = cd_fast.enet_coordinate_descent(
D:\anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.739e+02, tolerance: 2.810e+00
  model = cd_fast.enet_coordinate_descent(
D:\anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.412e+02, tolerance: 2.892e+00
  model = cd_fast.enet_coo

,estimator,Pipeline(step...iter=50000))])
,param_grid,"{'lasso__alpha': [0.001, 0.01, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,degree,10


In [98]:
print(f"Best alpha (Lasso): {grid_lasso.best_params_}")
print(f"Test R²: {r2_score(y_test, grid_lasso.predict(X_test)):.4f}")

Best alpha (Lasso): {'lasso__alpha': 0.01}
Test R²: 0.9701
